# Masknmf: ASAP7, TZ_SCE014 MUnit_7, green channel

In [1]:
from pathlib import Path

import mbo_utilities as mbo
import masknmf

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),NVIDIA GeForce RTX 5060,DiscreteGPU,Vulkan,596.49
✅,NVIDIA GeForce RTX 5060,DiscreteGPU,D3D12,
❗ limited,Microsoft Basic Render Driver,CPU,D3D12,
❌,NVIDIA GeForce RTX 5060/PCIe/SSE2,Unknown,OpenGL,4.6.0 NVIDIA 596.49


In [2]:
MESC_PATH = Path("~/data/todd/temp/TZ_SCE014_01_1green_1red_zstack_ref_20260929.mesc").expanduser()
UNIT = "MSession_0/MUnit_7"
CHANNEL = 0
EXCLUDE_INITIAL_FRAMES = 200
OUTPUT_FOLDER = MESC_PATH.parent / "masknmf_asap7" / UNIT.replace("/", "_")
DEVICE = "cuda"
MAD_CORRELATION_THRESHOLD = 0.6

In [3]:
arr = mbo.imread(MESC_PATH, unit=UNIT)
raw = arr[EXCLUDE_INITIAL_FRAMES:, CHANNEL, 0]
movie = masknmf.OphysArray(raw, negative_indicator=True, include_mean=True, device=DEVICE)
movie.shape

(46097, 33, 168)

In [4]:
# 1. Compress raw data
compressor = masknmf.CompressStrategy(block_sizes=[4, 4],
                                      max_components=20,
                                      max_consecutive_failures=1,
                                      temporal_avg_factor=2,
                                      spatial_avg_factor=4,
                                      frame_batch_size=300)
raw_compressed = compressor.compress(movie)
raw_compressed.to(DEVICE)

# 2. Compute shifts on compressed/denoised data
corrector = masknmf.RigidMotionCorrector(max_shifts=[40, 40])
corrector.compute_template(raw_compressed)

# 3. Apply shifts to the raw data
registered = corrector.motion_correct(reference_movie=raw_compressed, target_movie=movie)

# 4. denoise the newly registered data (for visualization purposes only)
denoiser = masknmf.CompressDenoiseStrategy(block_sizes=[4, 4],
                                           max_components=20,
                                           max_consecutive_failures=1,
                                           temporal_avg_factor=2,
                                           spatial_avg_factor=4,
                                           frame_batch_size=300)
denoised = denoiser.compress(registered)
denoised.to(DEVICE)

[20261007T115211]: Starting compression
[20261007T115211]: sampled from the following regions: [0]
[20261007T115211]: We are initializing on a total of 46097 frames
[20261007T115211]: Loading data to estimate complete spatial basis
[20261007T115211]: skipping the pruning step for frame cutoff
[20261007T115211]: Finding spatiotemporal roughness thresholds


100%|██████████| 250/250 [00:01<00:00, 179.92it/s]

[20261007T115212]: Running Blockwise Decompositions



100%|██████████| 16/16 [00:21<00:00,  1.32s/it]

[20261007T115233]: Constructed spatial_compressed matrix. Rank of spatial_compressed is 1328
[20261007T115233]: Compression Object constructed


...

[20261007T120735]: Starting compression
[20261007T120735]: sampled from the following regions: [0]
[20261007T120735]: We are initializing on a total of 46097 frames
[20261007T120735]: Loading data to estimate complete spatial basis
[20261007T120735]: skipping the pruning step for frame cutoff
[20261007T120735]: Finding spatiotemporal roughness thresholds


100%|██████████| 250/250 [00:51<00:00,  4.85it/s]

[20261007T120827]: Running Blockwise Decompositions



100%|██████████| 16/16 [09:39<00:00, 36.21s/it]

[20261007T121806]: Constructed spatial_compressed matrix. Rank of spatial_compressed is 1328
[20261007T121806]: Compression Object constructed


In [ ]:
import os
os.getcwd()

'c:\\Users\\loson\\envs\\pml'

In [ ]:
registered.export("../registered.hdf5")

Help on method export in module masknmf.motion_correction.registration_arrays:

export(path: str | pathlib.Path, prefix: str = '') method of masknmf.motion_correction.rigid_strategy.RigidRegistrationArray instance
    Export to an HDF5 file, as the group named after this class.
    Requires ``h5py`` http://docs.h5py.org/

    Args:
        path (str): Full file path. Created if missing; in an existing file the other groups are kept
            and this class's group is replaced, so one file can hold every stage of a pipeline.
        prefix (str): Parent group, e.g. "calcium" writes "calcium/<class name>"; empty writes at the root.



In [14]:
import fastplotlib as fpl
from masknmf.visualization.imgui import TracePlot

panels = {
    "raw": movie,
    "pmd(raw)": raw_compressed,
    "registered": registered,
    "pmd(registered)": denoised,
}
ndw = fpl.NDWidget(names=list(panels), shape=(2, 2), size=(900, 700))
for name, data in panels.items():
    image = ndw[name].add_nd_image(data, ["time", "m", "n"], ["m", "n"], spatial_func=None, name=name)
    image.graphic.cmap = "gray"

shifts = registered.shifts.cpu().numpy()
traces = TracePlot(("shift (px)",), shifts.shape[0])
traces.set("shift (px)", [
    ("height", shifts[:, 0], (1.0, 0.6, 0.2)),
    ("width", shifts[:, 1], (0.4, 0.7, 1.0)),
])
traces.dock(ndw.figure, size=250, title="shifts")
traces.link(ndw.indices)

ndw.show()

In [ ]:
passes = []
for _ in range(2):
    init_config = masknmf.SuperpixelInitConfig(mad_correlation_threshold=MAD_CORRELATION_THRESHOLD,
                                               residual_threshold=0.1,
                                               sign="positive")
    nmf_config = masknmf.NMFConfig(support_threshold=(0.95, 0.7),
                                   ring_model_start_pt=41,
                                   min_brightness=0.0,
                                   merge_threshold=0.7,
                                   reassign_background=False)
    passes.append(masknmf.SinglepassDemixingConfig(init_config, nmf_config))
demixing_config = masknmf.MultipassDemixingConfig(passes)

pipeline = masknmf.GlutamateCalciumSpinePipeline(output_folder=OUTPUT_FOLDER,
                                                 demixing_config=demixing_config,
                                                 device=DEVICE)
run_folder = pipeline.run(glutamate_channel=denoised,
                          calcium_channel=None,
                          exclude_initial_frames=0)
run_folder